## Configuration Parameters

Configure all dataset sources, sampling parameters, and Google Earth Engine (GEE) settings below.


In [1]:
# ============================================================================
# IMPORTS
# ============================================================================

# Core data manipulation
import pandas as pd
import numpy as np
import geopandas as gpd
import pyarrow.parquet as pq

# Date and time handling
from datetime import datetime, timedelta

# File and path handling
import os
import sys
import json
from pathlib import Path

# Utilities
import warnings
import time
from typing import Dict, List, Optional, Tuple

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ============================================================================
# DIRECTORY DEFINITIONS
# ============================================================================

# Define data directories
RAW_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
MODEL_DIR = Path('models')

# Create directories if they don't exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("✓ All imports loaded successfully")
print(f"✓ Data directories initialized:")
print(f"  - Raw: {RAW_DIR}")
print(f"  - Processed: {PROCESSED_DIR}")
print(f"  - Models: {MODEL_DIR}")


✓ All imports loaded successfully
✓ Data directories initialized:
  - Raw: data\raw
  - Processed: data\processed
  - Models: models


In [2]:
# ============================================================================
# DATASET CONFIGURATION
# ============================================================================
# 
# ACTIVE PIPELINE: GEE-Based (fires + weather from Google Earth Engine)
#   ✓ Solves date alignment issues
#   ✓ Historical data (2000-2024)
#   ✓ Better data quality
# 
# To switch back to legacy pipeline:
#   - Set USE_MODIS_GEE = False and USE_MODIS_SHAPEFILE = True
#   - Set USE_GEE_WEATHER = False and USE_NASA_POWER = True
# ============================================================================

# --- Sample Size Configuration ---
SAMPLE_SIZE = 3000  # Number of fire samples to process (3000 for medium-scale training)
RANDOM_STATE = 42   # Random seed for reproducibility

# --- MODIS Fire Detection Data ---
# PRIMARY METHOD: Choose ONE fire detection source
USE_MODIS_GEE = True  # ← Use GEE for fire detection (RECOMMENDED - historical dates)
USE_MODIS_SHAPEFILE = False  # ← Use local shapefile (DEPRECATED - causes date issues)
MODIS_SHAPEFILE = 'raw_data/modis_fire_data/MODIS_C6_1_Global_24h.shp'

# GEE Fire Detection Configuration
GEE_FIRE_DATE_START = '2023-01-01'  # Start date for fire detection
GEE_FIRE_DATE_END = '2023-12-31'    # End date for fire detection
GEE_FIRE_PRODUCT = 'MODIS/006/MOD14A1'  # MOD14A1 (Terra) or MYD14A1 (Aqua) or FIRMS
GEE_FIRE_MIN_CONFIDENCE = 80  # Minimum fire detection confidence (0-100)
GEE_FIRE_MIN_FRP = 0  # Minimum Fire Radiative Power (MW)

# --- Weather Data Configuration ---
# PRIMARY METHOD: Choose ONE weather source
USE_GEE_WEATHER = True  # ← Use GEE ERA5 (RECOMMENDED - no date issues, better quality)
USE_NASA_POWER = False  # ← Use NASA POWER API (DEPRECATED - HTTP 422 errors with future dates)

# GEE ERA5 Weather Configuration
GEE_WEATHER_PRODUCT = 'ECMWF/ERA5_LAND/DAILY_AGGR'  # ERA5-Land daily aggregated
GEE_WEATHER_BANDS = [
    'temperature_2m',           # 2m temperature (K)
    'temperature_2m_max',       # Daily max temperature
    'temperature_2m_min',       # Daily min temperature
    'dewpoint_temperature_2m',  # Dewpoint (K) - for humidity
    'total_precipitation_sum',  # Total precipitation (m)
    'u_component_of_wind_10m',  # U wind component (m/s)
    'v_component_of_wind_10m',  # V wind component (m/s)
    'surface_pressure'          # Surface pressure (Pa)
]
WEATHER_LOOKBACK_DAYS = 14  # Days before fire event to fetch weather data

# NASA POWER (legacy - only if USE_NASA_POWER = True)
NASA_POWER_API_KEY = None
NASA_POWER_TEMPORAL = 'daily'
NASA_POWER_PARAMS = 'RH2M,RH2M_MAX,RH2M_MIN,PRECTOT,WS10M,WS10M_MAX,T2M,T2M_MAX,T2M_MIN,PS'

# --- Google Earth Engine (GEE) Configuration ---
USE_GEE = True  # Enable GEE for terrain, vegetation, and fuel data
GEE_PROJECT = 'ee-fireprediction'  # Your GEE project ID (or use environment variable)
GEE_KEY_PATH = None  # Will be loaded from environment variable 'GEE_KEY'
GEE_SERVICE_ACCOUNT = None  # Will be loaded from GEE credentials JSON
GEE_USE_SERVICE_ACCOUNT = True  # Use service account (True) or user auth (False)

# GEE Data Sources to Extract
GEE_EXTRACT_TERRAIN = True      # SRTM DEM: elevation, slope, aspect, curvature, ruggedness
GEE_EXTRACT_WATER = True        # Distance to water bodies (JRC Global Surface Water)
GEE_EXTRACT_VEGETATION = True   # NDVI, EVI, LAI from MODIS
GEE_EXTRACT_FUEL = True         # Fuel moisture, fuel type, vegetation density
GEE_EXTRACT_BIOME = True        # Biome classification (WWF Terrestrial Ecoregions)

# GEE Feature Format Configuration
GEE_FEATURE_FORMAT = {
    'terrain': {
        'elevation': 'float32',
        'slope': 'float32',
        'aspect': 'float32',
        'curvature': 'float32',
        'ruggedness': 'float32'
    },
    'water': {
        'distance_to_water': 'float32',
        'water_occurrence': 'float32'
    },
    'vegetation': {
        'ndvi': 'float32',
        'ndvi_14day_mean': 'float32',
        'ndvi_14day_std': 'float32',
        'evi': 'float32',
        'lai': 'float32'
    },
    'fuel': {
        'fuel_moisture_1000hr': 'float32',
        'fuel_type': 'int8',
        'vegetation_density': 'float32'
    },
    'biome': {
        'biome_id': 'int16',
        'biome_name': 'string',
        'realm': 'string'
    }
}

# --- Stratified Sampling Configuration ---
ENABLE_GEOGRAPHIC_STRATIFICATION = True  # Ensure diversity across lat/lon grid
N_LAT_BINS = 6   # Number of latitude bins for stratification
N_LON_BINS = 12  # Number of longitude bins for stratification (72 total regions)

ENABLE_TEMPORAL_STRATIFICATION = False  # TODO: Enable when multi-temporal MODIS data available
N_TEMPORAL_BINS = 4  # Quarters/seasons for temporal diversity

ENABLE_BIOME_STRATIFICATION = True  # Ensure diversity across biome types
MIN_UNIQUE_BIOMES = 5  # Minimum number of unique biomes required
MAX_BIOME_PROPORTION = 0.50  # Maximum proportion of samples from single biome

# --- Data Quality Requirements ---
MIN_HUMIDITY_COVERAGE = 0.80  # Minimum 80% of samples must have humidity data
MIN_PRECIPITATION_COVERAGE = 0.80  # Minimum 80% of samples must have precipitation data
MIN_CONFIDENCE_SCORE = 30  # Minimum MODIS confidence score (0-100)

# --- API Rate Limiting ---
NASA_POWER_API_DELAY = 0.1  # Seconds between requests (with API key)
NASA_POWER_MAX_CONCURRENT = 5  # Max concurrent requests (with API key)
GEE_BATCH_SIZE = 50  # Number of locations to batch in GEE requests

# --- Output Paths ---
OUTPUT_DIR = 'data/processed'
WEATHER_OUTPUT = f'{OUTPUT_DIR}/weather_features.parquet'
TERRAIN_OUTPUT = f'{OUTPUT_DIR}/terrain_features.parquet'
GEOSPATIAL_OUTPUT = f'{OUTPUT_DIR}/geospatial_features.parquet'
ML_READY_OUTPUT = f'{OUTPUT_DIR}/ml_ready.parquet'

# ============================================================================
# DISPLAY CONFIGURATION
# ============================================================================

print("="*80)
print("FIRE PREDICTION DATA INGESTION - CONFIGURATION")
print("="*80)

# Show active pipeline
pipeline_mode = "GEE-BASED" if (USE_MODIS_GEE and USE_GEE_WEATHER) else "LEGACY/MIXED"
print(f"\n🔧 ACTIVE PIPELINE: {pipeline_mode}")
if USE_MODIS_GEE and USE_GEE_WEATHER:
    print(f"   ✓ Using Google Earth Engine for fire detection and weather")
    print(f"   ✓ Historical data with perfect temporal alignment")
else:
    print(f"   ⚠️  Using legacy/mixed data sources")
    if USE_MODIS_SHAPEFILE:
        print(f"   ⚠️  Shapefile may have limited date range")
    if USE_NASA_POWER:
        print(f"   ⚠️  NASA POWER may have date alignment issues")

print(f"\n📊 Dataset Configuration:")
print(f"   Sample Size: {SAMPLE_SIZE}")
print(f"   Random State: {RANDOM_STATE}")
print(f"\n🔥 MODIS Fire Data:")
print(f"   GEE Source: {USE_MODIS_GEE}")
if USE_MODIS_GEE:
    print(f"   Product: {GEE_FIRE_PRODUCT}")
    print(f"   Date Range: {GEE_FIRE_DATE_START} to {GEE_FIRE_DATE_END}")
    print(f"   Min Confidence: {GEE_FIRE_MIN_CONFIDENCE}%")
    print(f"   Min FRP: {GEE_FIRE_MIN_FRP} MW")
else:
    print(f"   Shapefile: {MODIS_SHAPEFILE}")
    print(f"   Min Confidence: {MIN_CONFIDENCE_SCORE}")
print(f"\n🌤️  Weather Data:")
print(f"   GEE ERA5: {USE_GEE_WEATHER}")
if USE_GEE_WEATHER:
    print(f"   Product: {GEE_WEATHER_PRODUCT}")
    print(f"   Bands: {len(GEE_WEATHER_BANDS)} variables")
    print(f"   Lookback: {WEATHER_LOOKBACK_DAYS} days")
print(f"   NASA POWER: {USE_NASA_POWER}")
if USE_NASA_POWER:
    print(f"   Temporal: {NASA_POWER_TEMPORAL}")
    print(f"   Parameters: {NASA_POWER_PARAMS[:50]}...")
print(f"\n🌍 Google Earth Engine:")
print(f"   Enabled: {USE_GEE}")
print(f"   Extract Terrain: {GEE_EXTRACT_TERRAIN}")
print(f"   Extract Water: {GEE_EXTRACT_WATER}")
print(f"   Extract Vegetation: {GEE_EXTRACT_VEGETATION}")
print(f"   Extract Fuel: {GEE_EXTRACT_FUEL}")
print(f"   Extract Biome: {GEE_EXTRACT_BIOME}")
print(f"\n📍 Stratification:")
print(f"   Geographic: {ENABLE_GEOGRAPHIC_STRATIFICATION} ({N_LAT_BINS}x{N_LON_BINS} grid = {N_LAT_BINS*N_LON_BINS} regions)")
print(f"   Temporal: {ENABLE_TEMPORAL_STRATIFICATION} ({N_TEMPORAL_BINS} bins)")
print(f"   Biome: {ENABLE_BIOME_STRATIFICATION} (min {MIN_UNIQUE_BIOMES} types, max {MAX_BIOME_PROPORTION*100}% per type)")
print(f"\n✅ Quality Requirements:")
print(f"   Humidity Coverage: ≥{MIN_HUMIDITY_COVERAGE*100}%")
print(f"   Precipitation Coverage: ≥{MIN_PRECIPITATION_COVERAGE*100}%")
print(f"\n⚡ API Configuration:")
print(f"   NASA POWER Delay: {NASA_POWER_API_DELAY}s")
print(f"   NASA POWER Max Concurrent: {NASA_POWER_MAX_CONCURRENT}")
print(f"   GEE Batch Size: {GEE_BATCH_SIZE}")
print(f"\n💾 Output:")
print(f"   Directory: {OUTPUT_DIR}")
print(f"   ML-Ready Dataset: {ML_READY_OUTPUT}")
print("="*80)


FIRE PREDICTION DATA INGESTION - CONFIGURATION

🔧 ACTIVE PIPELINE: GEE-BASED
   ✓ Using Google Earth Engine for fire detection and weather
   ✓ Historical data with perfect temporal alignment

📊 Dataset Configuration:
   Sample Size: 3000
   Random State: 42

🔥 MODIS Fire Data:
   GEE Source: True
   Product: MODIS/006/MOD14A1
   Date Range: 2023-01-01 to 2023-12-31
   Min Confidence: 80%
   Min FRP: 0 MW

🌤️  Weather Data:
   GEE ERA5: True
   Product: ECMWF/ERA5_LAND/DAILY_AGGR
   Bands: 8 variables
   Lookback: 14 days
   NASA POWER: False

🌍 Google Earth Engine:
   Enabled: True
   Extract Terrain: True
   Extract Water: True
   Extract Vegetation: True
   Extract Fuel: True
   Extract Biome: True

📍 Stratification:
   Geographic: True (6x12 grid = 72 regions)
   Temporal: False (4 bins)
   Biome: True (min 5 types, max 50.0% per type)

✅ Quality Requirements:
   Humidity Coverage: ≥80.0%
   Precipitation Coverage: ≥80.0%

⚡ API Configuration:
   NASA POWER Delay: 0.1s
   NASA POWE

## Configuration

Set your configuration parameters here:


In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these parameters as needed
# ============================================================================

# Sample size: Number of fire detections to process (None = process all)
SAMPLE_SIZE = 1000  # Start with 100 for testing, increase for full dataset

# Date range for weather data: Days before fire detection to fetch
DAYS_BEFORE_FIRE = 14  # For 14-day fuel conditioning index

# Geographic filtering: Process all locations or filter by bounds?
# Set to None to process all locations, or specify [min_lon, min_lat, max_lon, max_lat]
GEOGRAPHIC_BOUNDS = None  # Example: [-180, -90, 180, 90] for global

# Data store format: 'parquet' (recommended), 'hdf5', or 'csv'
DATA_STORE_FORMAT = 'parquet'

# ============================================================================
# SMART RATE LIMITING FOR NASA POWER API
# ============================================================================
# NASA POWER API limits:
#   - Without API key (DEMO_KEY): 30 requests/hour, 50/day
#   - With personal API key: 1,000 requests/hour (FREE!)
# Get a FREE API key at: https://api.nasa.gov/
# Set environment variable: NASA_POWER_API_KEY=your_key_here

import os
NASA_API_KEY = os.getenv('NASA_POWER_API_KEY')

if NASA_API_KEY:
    # With API key: 1000 req/hr = ~3.6 sec minimum between requests
    # Use adaptive delay with concurrent requests for speed
    API_DELAY = 0.1  # Short delay between batches
    RATE_LIMIT_DELAY = 3.6  # Delay to respect 1000/hr limit
    MAX_CONCURRENT = 5  # Number of concurrent requests
    REQUESTS_PER_HOUR = 1000
    print("✓ NASA POWER API key detected - using optimized rate limits")
    print(f"  Rate: Up to {REQUESTS_PER_HOUR} requests/hour")
else:
    # Without API key: 30 req/hr = 120 sec between requests!
    API_DELAY = 120  # 2 minutes between requests (30/hr limit)
    RATE_LIMIT_DELAY = 120
    MAX_CONCURRENT = 1  # No concurrent requests allowed
    REQUESTS_PER_HOUR = 30
    print("⚠️  No NASA POWER API key found!")
    print("   Using DEMO_KEY: limited to 30 requests/hour (2 min delay)")
    print("   ⏱️  Estimated time for 3000 samples: ~100 hours!")
    print("")
    print("   🚀 GET FREE API KEY (5 min setup):")
    print("   1. Go to: https://api.nasa.gov/")
    print("   2. Sign up for free API key")
    print("   3. Set environment variable: NASA_POWER_API_KEY=your_key")
    print("   4. Re-run this notebook")
    print("")
    print("   With API key: 3000 samples in ~3 hours (33x faster!)")

# Batch processing: Process features in batches
BATCH_SIZE = 10  # Process 10 fire detections at a time

# Feature flags: Which features to fetch
FETCH_WEATHER = True
FETCH_TERRAIN = True
FETCH_WATER_DISTANCE = True
FETCH_VEGETATION = True

# Terrain data source: 'GEE' (Google Earth Engine) or 'NASA_DEM' (OpenTopography API)
# GEE is recommended (no rate limits, faster, better coverage)
USE_GEE_TERRAIN = True  # Set to False to use NASA DEM API instead

print("Configuration:")
print(f"  Sample size: {SAMPLE_SIZE if SAMPLE_SIZE else 'All'}")
print(f"  Days before fire: {DAYS_BEFORE_FIRE}")
print(f"  Geographic bounds: {GEOGRAPHIC_BOUNDS if GEOGRAPHIC_BOUNDS else 'All locations'}")
print(f"  Data format: {DATA_STORE_FORMAT}")
print(f"\nTerrain Data Source:")
print(f"  Use GEE: {USE_GEE_TERRAIN if 'USE_GEE_TERRAIN' in globals() else 'Not set'}")
print(f"  API delay: {API_DELAY}s")
print(f"  Batch size: {BATCH_SIZE}")


✓ NASA POWER API key detected - using optimized rate limits
  Rate: Up to 1000 requests/hour
Configuration:
  Sample size: 1000
  Days before fire: 14
  Geographic bounds: All locations
  Data format: parquet
  API delay: 0.1s
  Batch size: 10


In [4]:
# ============================================================================
# GOOGLE EARTH ENGINE INITIALIZATION
# ============================================================================

if USE_GEE or USE_MODIS_GEE or USE_GEE_WEATHER:
    import ee
    
    print("="*80)
    print("INITIALIZING GOOGLE EARTH ENGINE")
    print("="*80)
    
    try:
        # Get project from environment variable if not set
        project = os.getenv('GEE_PROJECT', GEE_PROJECT)
        
        if GEE_USE_SERVICE_ACCOUNT:
            # Service Account Authentication (recommended for automation)
            print("\nUsing Service Account authentication...")
            
            # Get credentials path from environment
            gee_key_path = os.getenv('GEE_KEY', GEE_KEY_PATH)
            
            if not gee_key_path:
                raise ValueError("GEE_KEY environment variable not set. Please set it to your service account JSON path.")
            
            # Clean up path
            gee_key_path = gee_key_path.strip('"\'')
            gee_key_path = ''.join(c for c in gee_key_path if ord(c) >= 32 or c in '\\\\/')
            gee_key_path = gee_key_path.replace('/', '\\\\')
            gee_key_path = gee_key_path.strip()
            
            # Check if file exists
            if not os.path.isfile(gee_key_path):
                # Try to find in project directory
                print(f"  Credentials not found at {gee_key_path}")
                print(f"  Searching in project directory...")
                for jf in Path('.').glob('*.json'):
                    if any(keyword in jf.name.lower() for keyword in ['service', 'gee', 'fire', 'earth']):
                        gee_key_path = str(jf.resolve())
                        print(f"  Found credentials: {gee_key_path}")
                        break
            
            if not os.path.isfile(gee_key_path):
                raise FileNotFoundError(f"Cannot find GEE credentials file: {gee_key_path}")
            
            print(f"✓ Using credentials: {gee_key_path}")
            
            # Load service account email
            with open(gee_key_path, 'r') as f:
                key_data = json.load(f)
            
            service_account_email = key_data.get('client_email')
            if not service_account_email:
                raise ValueError("No client_email found in credentials JSON")
            
            # Get project from credentials if not specified
            if not project or project == 'ee-fireprediction':
                project = key_data.get('project_id', project)
            
            print(f"✓ Service account: {service_account_email}")
            print(f"✓ Project: {project}")
            
            # Initialize with service account
            credentials = ee.ServiceAccountCredentials(service_account_email, gee_key_path)
            ee.Initialize(credentials)
            
        else:
            # User Authentication (interactive)
            print("\nUsing User authentication...")
            print("If this is your first time, you'll need to authenticate in your browser.")
            
            try:
                # Try to initialize (will use cached credentials if available)
                ee.Initialize(project=project)
                print(f"✓ Initialized with cached credentials")
            except Exception as e:
                # Need to authenticate
                print(f"✓ Authentication required...")
                ee.Authenticate()
                ee.Initialize(project=project)
                print(f"✓ Authentication successful")
            
            print(f"✓ Project: {project}")
        
        # Test the connection
        test = ee.Number(1).getInfo()
        print(f"\n✅ Google Earth Engine initialized successfully!")
        print(f"   Ready for fire detection and weather data extraction.")
        
    except Exception as e:
        print(f"\n❌ Error initializing Google Earth Engine: {e}")
        print(f"\nTroubleshooting:")
        print(f"  1. Make sure you have earthengine-api installed: pip install earthengine-api")
        print(f"  2. For service account: Set GEE_KEY environment variable to your JSON key path")
        print(f"  3. For user auth: Run ee.Authenticate() first or set GEE_USE_SERVICE_ACCOUNT=False")
        print(f"  4. Make sure your GEE project ID is correct: {project}")
        raise
    
    print("="*80)
    
else:
    print("Google Earth Engine is disabled in configuration.")


INITIALIZING GOOGLE EARTH ENGINE

Using Service Account authentication...
✓ Using credentials: C:\Users\Drewo\OneDrive\Documents\GIT\fire_prediction\fireprediction-483622-3e2ab1191a16.json
✓ Service account: acount-1@fireprediction-483622.iam.gserviceaccount.com
✓ Project: fireprediction-483622

✅ Google Earth Engine initialized successfully!
   Ready for fire detection and weather data extraction.


## Step 1: Load MODIS Fire Detection Data

Load the raw fire detection data from the shapefile.


In [5]:
# ============================================================================
# STEP 1A: FETCH MODIS FIRE DETECTIONS FROM GEE
# ============================================================================

if USE_MODIS_GEE:
    print("="*80)
    print("FETCHING MODIS FIRE DETECTIONS FROM GOOGLE EARTH ENGINE")
    print("="*80)
    
    print(f"\nConfiguration:")
    print(f"  Products: MODIS/006/MOD14A1 (Terra) + MODIS/006/MYD14A1 (Aqua) - Combined")
    print(f"  Date Range: {GEE_FIRE_DATE_START} to {GEE_FIRE_DATE_END}")
    print(f"  Min Confidence: FireMask >= 8 (nominal + high confidence)")
    print(f"  Target Samples: {SAMPLE_SIZE}")
    print(f"  Extraction Method: reduceToVectors (extracts ALL fire pixels, not random sample)")
    
    # Load MODIS fire product - combine Terra and Aqua for better coverage
    # This gives us more historical fire detections for training
    fires_terra = ee.ImageCollection('MODIS/006/MOD14A1') \
        .filterDate(GEE_FIRE_DATE_START, GEE_FIRE_DATE_END)
    fires_aqua = ee.ImageCollection('MODIS/006/MYD14A1') \
        .filterDate(GEE_FIRE_DATE_START, GEE_FIRE_DATE_END)
    fires = fires_terra.merge(fires_aqua)
    
    # Check collection size
    collection_size = fires.size().getInfo()
    print(f"\n✓ Loaded fire collection: {collection_size} images (Terra + Aqua combined)")
    
    if collection_size == 0:
        raise ValueError(f"No images found in MODIS collections for date range {GEE_FIRE_DATE_START} to {GEE_FIRE_DATE_END}")
    
    # Function to extract fire pixels as features
    def extract_fire_points(image):
        # Get FireMask band (7-9 = fire detected, 8-9 = high confidence)
        fire_mask = image.select('FireMask')
        max_frp = image.select('MaxFRP')
        
        # Filter for high confidence fires (FireMask >= 8 for nominal+ confidence)
        # For training data, we want comprehensive coverage, so use >= 8
        # (8 = nominal confidence, 9 = high confidence)
        # Change to .eq(9) if you only want highest confidence fires
        high_confidence = fire_mask.gte(8)  # 8 or 9 = nominal or high confidence
        
        # Mask the image to only fire pixels
        masked = image.updateMask(high_confidence)
        
        # Use reduceToVectors to get ALL fire pixels, not random sample
        # This is critical for training data - we want comprehensive historical coverage
        # IMPORTANT: reduceToVectors expects a single-band image for the default Reducer.countEvery
        # So we reduce only the FireMask band here
        fire_vectors = masked.select(['FireMask']).reduceToVectors(
            geometry=ee.Geometry.Rectangle([-180, -90, 180, 90], None, False),
            # Use a slightly coarser scale and bestEffort to avoid maxPixels errors
            scale=2000,  # 2km resolution reduces total pixel count
            geometryType='centroid',  # Get point locations (centroids of fire pixels)
            maxPixels=1e9,
            bestEffort=True,
            crs='EPSG:4326'
        )
        
        # Add acquisition date and properties to each feature
        # Note: reduceToVectors on a single band will attach that band value as a property
        def add_properties(feature):
            return feature.set({
                'ACQ_DATE': image.date().format('YYYY-MM-dd'),
                'FireMask': feature.get('FireMask')
                # If you later need FRP, you can join it from a separate reduceRegion step
            })
        
        return fire_vectors.map(add_properties)
    
    print(f"Processing fire detections...")
    
    # Extract all fire points
    fire_points = fires.map(extract_fire_points).flatten()
    
    # Get total count for debugging
    try:
        total_count = fire_points.size().getInfo()
        print(f"✓ Total fire points in collection: {total_count}")
    except Exception as e:
        print(f"⚠ Could not get total count: {e}")
        total_count = None
    
    # Sample to get desired number of fires (or all if less than limit)
    sample_limit = min(SAMPLE_SIZE * 3, 50000) if total_count is None else min(SAMPLE_SIZE * 3, total_count, 50000)
    fire_sample = fire_points.limit(sample_limit)
    
    print(f"Attempting to retrieve up to {sample_limit} fire detections...")
    
    # Convert to list for local processing
    fire_list = fire_sample.getInfo()['features']
    
    print(f"✓ Retrieved {len(fire_list)} fire detections from GEE")
    
    # Debug: Print sample feature to see available properties
    if fire_list:
        print(f"\nDebug - Sample feature properties: {fire_list[0].get('properties', {})}")
        print(f"Debug - Sample feature keys: {list(fire_list[0].keys())}")
    
    # Convert to DataFrame
    fire_records = []
    for feature in fire_list:
        props = feature['properties']
        coords = feature['geometry']['coordinates']
        
        # Extract ACQ_DATE, handling both string and None values
        acq_date = props.get('ACQ_DATE')
        if acq_date is None:
            # Try to get date from image ID or other properties
            acq_date = props.get('date') or props.get('system:time_start')
        
        fire_records.append({
            'LONGITUDE': coords[0],
            'LATITUDE': coords[1],
            'ACQ_DATE': acq_date,
            'BRIGHTNESS': props.get('MaxFRP', 0),  # Using FRP as proxy
            'FRP': props.get('MaxFRP', 0),
            'CONFIDENCE': props.get('FireMask', 0),
            'SATELLITE': 'MODIS',
            'VERSION': '6.1',
            'DAYNIGHT': 'D'  # GEE doesn't provide this directly
        })
    
    fire_data = pd.DataFrame(fire_records)
    
    # Check if we have any data
    if len(fire_data) == 0:
        raise ValueError("No fire detections retrieved from GEE. Check your date range and filters.")
    
    # Check if ACQ_DATE column exists and has valid data
    if 'ACQ_DATE' not in fire_data.columns:
        print("⚠ WARNING: ACQ_DATE column not found in fire data. Checking available columns...")
        print(f"Available columns: {fire_data.columns.tolist()}")
        print(f"Sample properties: {fire_list[0]['properties'] if fire_list else 'No features'}")
        raise ValueError("ACQ_DATE column is missing from fire data")
    
    # Convert date to datetime, handling None values
    if fire_data['ACQ_DATE'].isna().all():
        print("⚠ WARNING: All ACQ_DATE values are None. Sample properties:", fire_list[0]['properties'] if fire_list else 'No features')
        raise ValueError("ACQ_DATE values are missing from all fire detections")
    
    # Drop rows with None dates and convert
    fire_data = fire_data.dropna(subset=['ACQ_DATE'])
    fire_data['ACQ_DATE'] = pd.to_datetime(fire_data['ACQ_DATE'])
    
    # Apply stratified sampling if enabled
    if ENABLE_GEOGRAPHIC_STRATIFICATION and len(fire_data) > SAMPLE_SIZE:
        print(f"\nApplying stratified geographic sampling...")
        
        # Create lat/lon bins
        fire_data['lat_bin'] = pd.cut(fire_data['LATITUDE'], bins=N_LAT_BINS, labels=False)
        fire_data['lon_bin'] = pd.cut(fire_data['LONGITUDE'], bins=N_LON_BINS, labels=False)
        
        # Sample from each bin
        sampled = fire_data.groupby(['lat_bin', 'lon_bin']).apply(
            lambda x: x.sample(min(len(x), max(1, SAMPLE_SIZE // (N_LAT_BINS * N_LON_BINS))), 
                             random_state=RANDOM_STATE)
        ).reset_index(drop=True)
        
        # If we don't have enough, sample more from larger bins
        if len(sampled) < SAMPLE_SIZE:
            remaining = SAMPLE_SIZE - len(sampled)
            extra = fire_data[~fire_data.index.isin(sampled.index)].sample(
                min(remaining, len(fire_data) - len(sampled)), 
                random_state=RANDOM_STATE
            )
            fire_data = pd.concat([sampled, extra]).reset_index(drop=True)
        else:
            fire_data = sampled.sample(min(len(sampled), SAMPLE_SIZE), random_state=RANDOM_STATE)
        
        fire_data = fire_data.drop(['lat_bin', 'lon_bin'], axis=1)
    elif len(fire_data) > SAMPLE_SIZE:
        fire_data = fire_data.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
    
    print(f"\n✓ Final dataset: {len(fire_data)} fire detections")
    print(f"  Date range: {fire_data['ACQ_DATE'].min()} to {fire_data['ACQ_DATE'].max()}")
    print(f"  Lat range: {fire_data['LATITUDE'].min():.2f} to {fire_data['LATITUDE'].max():.2f}")
    print(f"  Lon range: {fire_data['LONGITUDE'].min():.2f} to {fire_data['LONGITUDE'].max():.2f}")
    
    # Save to parquet
    output_path = RAW_DIR / 'fire_detections_gee.parquet'
    fire_data.to_parquet(output_path, index=False)
    print(f"✓ Saved to {output_path}")
    
    print("="*80)
    
else:
    print("GEE fire detection disabled. Will use shapefile instead.")


FETCHING MODIS FIRE DETECTIONS FROM GOOGLE EARTH ENGINE

Configuration:
  Products: MODIS/006/MOD14A1 (Terra) + MODIS/006/MYD14A1 (Aqua) - Combined
  Date Range: 2023-01-01 to 2023-12-31
  Min Confidence: FireMask >= 8 (nominal + high confidence)
  Target Samples: 1000
  Extraction Method: reduceToVectors (extracts ALL fire pixels, not random sample)


c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MOD14A1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MOD14A1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD14A1

  warnings.warn(warning, category=DeprecationWarning)
c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MYD14A1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MYD14A1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MYD14A1

  warnings.warn(warning, category=DeprecationWarning)



✓ Loaded fire collection: 103 images (Terra + Aqua combined)
Processing fire detections...
✓ Total fire points in collection: 176429
Attempting to retrieve up to 3000 fire detections...
✓ Retrieved 3000 fire detections from GEE

Debug - Sample feature properties: {'ACQ_DATE': '2023-01-01', 'count': 1, 'label': 8}
Debug - Sample feature keys: ['type', 'geometry', 'id', 'properties']

Applying stratified geographic sampling...

✓ Final dataset: 1000 fire detections
  Date range: 2023-01-01 00:00:00 to 2023-01-03 00:00:00
  Lat range: -40.06 to 59.14
  Lon range: -117.96 to 151.82
✓ Saved to data\raw\fire_detections_gee.parquet


In [ ]:
# ============================================================================
# STEP 2: FETCH ERA5 WEATHER DATA FROM GEE
# ============================================================================

# Check if weather data was already fetched
weather_already_fetched = (
    ('weather_df_all' in locals() or 'weather_df_all' in globals()) or
    (PROCESSED_DIR / 'weather_features_era5.parquet').exists() or
    (PROCESSED_DIR / 'weather_features.parquet').exists()
)

if USE_GEE_WEATHER and 'fire_data' in locals() and not weather_already_fetched:
    print("="*80)
    print("FETCHING ERA5 WEATHER DATA FROM GOOGLE EARTH ENGINE")
    print("="*80)
    
    print(f"\nConfiguration:")
    print(f"  Product: {GEE_WEATHER_PRODUCT}")
    print(f"  Variables: {len(GEE_WEATHER_BANDS)}")
    print(f"  Lookback: {WEATHER_LOOKBACK_DAYS} days")
    print(f"  Total locations: {len(fire_data)}")
    
    # Load ERA5-Land daily aggregated data
    era5 = ee.ImageCollection(GEE_WEATHER_PRODUCT)
    
    weather_features = []
    
    print(f"\nProcessing weather data...")
    
    for idx, row in fire_data.iterrows():
        if idx % 100 == 0:
            print(f"  Processing {idx}/{len(fire_data)}...")
        
        lat = row['LATITUDE']
        lon = row['LONGITUDE']
        fire_date = pd.to_datetime(row['ACQ_DATE'])
        
        # Calculate date range
        start_date = (fire_date - timedelta(days=WEATHER_LOOKBACK_DAYS)).strftime('%Y-%m-%d')
        end_date = fire_date.strftime('%Y-%m-%d')
        
        # Filter ERA5 for this location and time period
        weather = era5.filterDate(start_date, end_date) \
            .select(GEE_WEATHER_BANDS)
        
        # Define point
        point = ee.Geometry.Point([lon, lat])
        
        try:
            # Extract values at point
            weather_series = weather.getRegion(point, 11132).getInfo()  # 11km scale for ERA5-Land
            
            # Parse the results
            if len(weather_series) > 1:
                headers = weather_series[0]
                data_rows = weather_series[1:]
                
                # Convert to DataFrame for easy aggregation
                weather_df = pd.DataFrame(data_rows, columns=headers)
                
                # Calculate features
                feature_dict = {
                    'fire_index': idx,
                    'latitude': lat,
                    'longitude': lon
                }
                
                # Temperature features (convert K to C)
                if 'temperature_2m' in headers:
                    temps = weather_df['temperature_2m'].astype(float) - 273.15
                    feature_dict['temp_mean'] = temps.mean()
                    feature_dict['temp_max'] = temps.max()
                    feature_dict['temp_min'] = temps.min()
                    feature_dict['temp_range'] = temps.max() - temps.min()
                
                # Temperature max/min
                if 'temperature_2m_max' in headers:
                    feature_dict['temp_extreme_max'] = weather_df['temperature_2m_max'].astype(float).max() - 273.15
                if 'temperature_2m_min' in headers:
                    feature_dict['temp_extreme_min'] = weather_df['temperature_2m_min'].astype(float).min() - 273.15
                
                # Humidity (from dewpoint)
                if 'dewpoint_temperature_2m' in headers and 'temperature_2m' in headers:
                    # Calculate relative humidity from dewpoint and temperature
                    T = weather_df['temperature_2m'].astype(float)
                    Td = weather_df['dewpoint_temperature_2m'].astype(float)
                    RH = 100 * (np.exp((17.625 * Td) / (243.04 + Td)) / np.exp((17.625 * T) / (243.04 + T)))
                    feature_dict['humidity_mean'] = RH.mean()
                    feature_dict['humidity_min'] = RH.min()
                
                # Precipitation (convert m to mm)
                if 'total_precipitation_sum' in headers:
                    precip = weather_df['total_precipitation_sum'].astype(float) * 1000
                    feature_dict['precip_total'] = precip.sum()
                    feature_dict['precip_mean'] = precip.mean()
                    feature_dict['precip_max'] = precip.max()
                    feature_dict['days_no_rain'] = (precip < 0.1).sum()
                
                # Wind (combine U and V components)
                if 'u_component_of_wind_10m' in headers and 'v_component_of_wind_10m' in headers:
                    u = weather_df['u_component_of_wind_10m'].astype(float)
                    v = weather_df['v_component_of_wind_10m'].astype(float)
                    wind_speed = np.sqrt(u**2 + v**2)
                    feature_dict['wind_speed_mean'] = wind_speed.mean()
                    feature_dict['wind_speed_max'] = wind_speed.max()
                
                # Pressure
                if 'surface_pressure' in headers:
                    pressure = weather_df['surface_pressure'].astype(float) / 100  # Pa to hPa
                    feature_dict['pressure_mean'] = pressure.mean()
                
                weather_features.append(feature_dict)
                
        except Exception as e:
            if idx % 100 == 0:
                print(f"    Warning: Failed to fetch weather for fire {idx}: {str(e)[:50]}")
            continue
    
    print(f"\n✓ Retrieved weather data for {len(weather_features)}/{len(fire_data)} locations")
    
    # Convert to DataFrame
    weather_df_all = pd.DataFrame(weather_features)
    
    if len(weather_df_all) > 0:
        print(f"\n✓ Weather features:")
        print(f"  Features: {len(weather_df_all.columns) - 3}")  # Excluding fire_index, lat, lon
        print(f"  Sample count: {len(weather_df_all)}")
        
        # Save weather features
        output_path = PROCESSED_DIR / 'weather_features_era5.parquet'
        weather_df_all.to_parquet(output_path, index=False)
        print(f"✓ Saved to {output_path}")
    else:
        print(f"✗ No weather features extracted")
    
    print("="*80)
    
elif weather_already_fetched:
    print("="*80)
    print("WEATHER DATA ALREADY FETCHED")
    print("="*80)
    print("✓ Weather features were already fetched in a previous cell.")
    print("  Loading from memory or saved file...")
    
    # Try to load from memory first
    if 'weather_df_all' in locals() or 'weather_df_all' in globals():
        weather_df_all = locals().get('weather_df_all') or globals().get('weather_df_all')
        print(f"✓ Loaded weather features from memory ({len(weather_df_all)} records)")
    else:
        # Load from saved file
        weather_path = PROCESSED_DIR / 'weather_features_era5.parquet'
        if not weather_path.exists():
            weather_path = PROCESSED_DIR / 'weather_features.parquet'
        if weather_path.exists():
            weather_df_all = pd.read_parquet(weather_path)
            print(f"✓ Loaded weather features from {weather_path} ({len(weather_df_all)} records)")
        else:
            print("⚠ Weather file not found, but flag indicates it was fetched")
    print("="*80)
    
else:
    if not USE_GEE_WEATHER:
        print("GEE ERA5 weather disabled. Will use NASA POWER instead.")
    else:
        print("No fire data available. Run fire detection first.")


FETCHING ERA5 WEATHER DATA FROM GOOGLE EARTH ENGINE

Configuration:
  Product: ECMWF/ERA5_LAND/DAILY_AGGR
  Variables: 8
  Lookback: 14 days
  Total locations: 1000

Processing weather data...
  Processing 0/1000...
  Processing 100/1000...
  Processing 200/1000...
  Processing 300/1000...
  Processing 400/1000...
  Processing 500/1000...
  Processing 600/1000...
  Processing 700/1000...
  Processing 800/1000...
  Processing 900/1000...

✓ Retrieved weather data for 1000/1000 locations

✓ Weather features:
  Features: 15
  Sample count: 1000
✓ Saved to data\processed\weather_features_era5.parquet


In [7]:
# Load MODIS fire detection shapefile
shapefile_path = 'data_ingest/modis_fire/MODIS_C6_1_Global_24h.shp'

try:
    print("Loading MODIS fire detection data...")
    fire_data = gpd.read_file(shapefile_path)
    
    print(f"✓ Successfully loaded {len(fire_data)} fire detections")
    
    # Apply geographic filtering if specified
    if GEOGRAPHIC_BOUNDS:
        min_lon, min_lat, max_lon, max_lat = GEOGRAPHIC_BOUNDS
        mask = (
            (fire_data['LONGITUDE'] >= min_lon) & 
            (fire_data['LONGITUDE'] <= max_lon) &
            (fire_data['LATITUDE'] >= min_lat) & 
            (fire_data['LATITUDE'] <= max_lat)
        )
        fire_data = fire_data[mask]
        print(f"✓ Filtered to {len(fire_data)} detections within bounds")
    
    # Sample if specified
    # Stratified geographic sampling for diversity
    def stratified_geographic_sampling(fire_data, sample_size, random_state=42):
        """
        Stratified sampling to ensure geographic diversity across different regions.
        Divides the world into geographic bins and samples proportionally from each.
        """
        import numpy as np
        
        fire_data = fire_data.copy()
        # Create geographic bins (lat/lon grid)
        # Divide latitude into 6 bins (roughly 30 degrees each)
        # Divide longitude into 12 bins (roughly 30 degrees each)
        n_lat_bins = 6
        n_lon_bins = 12
        
        fire_data['lat_bin'] = pd.cut(fire_data['LATITUDE'], bins=n_lat_bins, labels=False)
        fire_data['lon_bin'] = pd.cut(fire_data['LONGITUDE'], bins=n_lon_bins, labels=False)
        fire_data['geo_bin'] = fire_data['lat_bin'].astype(str) + '_' + fire_data['lon_bin'].astype(str)
        
        # Calculate samples per bin (proportional to bin size, with minimum)
        bin_counts = fire_data['geo_bin'].value_counts()
        total_fires = len(fire_data)
        min_per_bin = max(1, sample_size // (n_lat_bins * n_lon_bins))  # At least 1 per bin
        
        sampled_data = []
        remaining_samples = sample_size
        
        # Sample from each bin proportionally
        for geo_bin in bin_counts.index:
            bin_data = fire_data[fire_data['geo_bin'] == geo_bin]
            bin_size = len(bin_data)
            
            if bin_size == 0:
                continue
                
            # Proportional sampling with minimum
            bin_proportion = bin_size / total_fires
            bin_samples = max(min_per_bin, int(bin_proportion * sample_size))
            bin_samples = min(bin_samples, bin_size, remaining_samples)
            
            if bin_samples > 0:
                sampled = bin_data.sample(n=bin_samples, random_state=random_state)
                sampled_data.append(sampled)
                remaining_samples -= bin_samples
        
        # If we haven't reached sample_size, randomly sample from remaining data
        if remaining_samples > 0:
            all_sampled = pd.concat(sampled_data, ignore_index=True) if sampled_data else pd.DataFrame()
            remaining_data = fire_data[~fire_data.index.isin(all_sampled.index)]
            if len(remaining_data) > 0:
                additional = remaining_data.sample(n=min(remaining_samples, len(remaining_data)), 
                                               random_state=random_state)
                sampled_data.append(additional)
        
        result = pd.concat(sampled_data, ignore_index=True) if sampled_data else fire_data
        result = result.drop(columns=['lat_bin', 'lon_bin', 'geo_bin'], errors='ignore')
        
        print(f"  Geographic diversity: {result['geo_bin'].nunique() if 'geo_bin' in result.columns else len(result)} unique regions")
        return result
    
    # Sample if specified - use stratified sampling for geographic diversity
    if SAMPLE_SIZE and len(fire_data) > SAMPLE_SIZE:
        print(f"Applying stratified geographic sampling to select {SAMPLE_SIZE} samples...")
        fire_data = stratified_geographic_sampling(fire_data, SAMPLE_SIZE, random_state=42)
        print(f"✓ Sampled to {len(fire_data)} detections with geographic diversity")
    else:
        print(f"✓ Using all {len(fire_data)} detections")
    
    # Ensure ACQ_DATE is datetime
    if not pd.api.types.is_datetime64_any_dtype(fire_data['ACQ_DATE']):
        fire_data['ACQ_DATE'] = pd.to_datetime(fire_data['ACQ_DATE'])
    
    # Sort by date for consistent processing
    fire_data = fire_data.sort_values('ACQ_DATE').reset_index(drop=True)
    
    print(f"\nDate range: {fire_data['ACQ_DATE'].min()} to {fire_data['ACQ_DATE'].max()}")
    print(f"Geographic bounds: {fire_data.total_bounds}")
    
    # Check for future dates (NASA POWER doesn't have future data)
    today = pd.Timestamp.now()
    future_dates = fire_data[fire_data['ACQ_DATE'] > today]
    if len(future_dates) > 0:
        print(f"\n⚠ WARNING: {len(future_dates)} fire detections have future dates (after {today.date()})")
        print(f"  NASA POWER API does not have data for future dates.")
        print(f"  Weather features will be empty for these detections.")
        print(f"  Consider using historical fire data for testing.")
    
    # Save raw fire detection data
    output_path = RAW_DIR / 'fire_detections.parquet'
    fire_data.to_parquet(output_path, index=False)
    print(f"✓ Saved raw fire detections to {output_path}")
    
except Exception as e:
    print(f"✗ Error loading fire detection data: {e}")
    raise


Loading MODIS fire detection data...
✓ Successfully loaded 20731 fire detections
Applying stratified geographic sampling to select 1000 samples...
  Geographic diversity: 1000 unique regions
✓ Sampled to 1000 detections with geographic diversity

Date range: 2026-01-05 00:00:00 to 2026-01-06 00:00:00
Geographic bounds: [-108.80891  -35.98851  147.45329   35.48832]
✓ Saved raw fire detections to data\raw\fire_detections.parquet


## Step 2: Fetch Weather Features (NASA POWER)

Fetch weather data for each fire detection location, including:
- Relative humidity (RH2M) - for 3-day humid index and 14-day fuel conditioning
- Precipitation (PRECTOT) - for 3-day dry/wet index and weighted extremes
- Wind speed (WS10M) - for weighted weather extremes
- Temperature (T2M) - for soft binary threshold


*Below is a simple api testing script*


In [8]:
from data_ingest.nasa_power.get_humidity import fetch_fire_prediction_weather
import requests

print("="*80)
print("NASA POWER API TEST - Single Location")
print("="*80)

row = fire_data.iloc[0]
fire_date = pd.to_datetime(row['ACQ_DATE'])
start_date = (fire_date - timedelta(days=DAYS_BEFORE_FIRE)).strftime('%Y%m%d')
end_date = fire_date.strftime('%Y%m%d')

print(f"\nTest Parameters:")
print(f"  Location: ({row['LATITUDE']:.4f}, {row['LONGITUDE']:.4f})")
print(f"  Fire Date: {fire_date.strftime('%Y-%m-%d')}")
print(f"  Date Range: {start_date} to {end_date}")
print(f"  Days lookback: {DAYS_BEFORE_FIRE}")
print(f"\nNASA POWER Coverage:")
print(f"  Data available: 1981-01-01 to approximately {datetime.now().strftime('%Y-%m-%d')}")
print(f"  Your date is {'WITHIN' if fire_date <= datetime.now() else 'OUTSIDE'} coverage")

# Check if dates are in the future
today = datetime.now()
if fire_date > today:
    print(f"\n⚠️  WARNING: Fire date ({fire_date.strftime('%Y-%m-%d')}) is in the FUTURE!")
    print(f"  Current date: {today.strftime('%Y-%m-%d')}")
    print(f"  NASA POWER API only has historical data (up to ~{today.strftime('%Y-%m-%d')})")
    print(f"  Expected result: HTTP 422 error (date not available)")

print(f"\nMaking API request...")

try:
    raw = fetch_fire_prediction_weather(
        latitude=row['LATITUDE'],
        longitude=row['LONGITUDE'],
        start_date=start_date,
        end_date=end_date,
        units='metric',
        return_dataframe=False
    )
    
    print(f"\n✓ API Response received:")
    if isinstance(raw, dict):
        if 'error' in raw:
            print(f"  ✗ Error: {raw['error']}")
        else:
            print(f"  ✓ Success: {len(raw.get('properties', {}).get('parameter', {}).get('RH2M', {}))} data points")
            print(f"\nResponse structure:")
            for key in raw.keys():
                print(f"    - {key}")
    else:
        print(f"  Response type: {type(raw)}")
        print(f"  Response: {raw}")
        
except requests.exceptions.HTTPError as e:
    print(f"\n✗ HTTP Error: {e}")
    print(f"  Status Code: {e.response.status_code if hasattr(e, 'response') else 'Unknown'}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"  Response Text: {e.response.text[:500]}")
except Exception as e:
    print(f"\n✗ Error: {type(e).__name__}: {str(e)}")
    import traceback
    print(f"\nFull traceback:")
    traceback.print_exc()

print("="*80)

NASA POWER API TEST - Single Location

Test Parameters:
  Location: (8.8244, -0.8625)
  Fire Date: 2026-01-05
  Date Range: 20251222 to 20260105
  Days lookback: 14

NASA POWER Coverage:
  Data available: 1981-01-01 to approximately 2026-01-12
  Your date is WITHIN coverage

Making API request...
Error fetching data from NASA POWER API: 422 Client Error:  for url: https://power.larc.nasa.gov/api/temporal/daily/point?start=20251222&end=20260105&latitude=8.8244&longitude=-0.86251&community=ag&parameters=RH2M%2CRH2M_MAX%2CRH2M_MIN%2CPRECTOT%2CWS10M%2CWS10M_MAX%2CT2M%2CT2M_MAX%2CT2M_MIN%2CPS&format=json&units=metric&user=Drew&header=true&time-standard=lst&key=zJ9rJBeKhgb1DNXsgV847uZrgKU2PZerIdSYBTTp

✓ API Response received:
  Response type: <class 'pandas.core.frame.DataFrame'>
  Response: Empty DataFrame
Columns: []
Index: []


In [9]:
if FETCH_WEATHER:
    # Check if fire_data exists
    if 'fire_data' not in locals() and 'fire_data' not in globals():
        print("⚠ Error: 'fire_data' not found. Please run the fire detection cell first.")
        raise NameError("'fire_data' must be defined before fetching weather data. Run the fire detection cell first.")
    
    # Check if weather data was already fetched in STEP 2
    weather_already_fetched = (
        ('weather_df_all' in locals() or 'weather_df_all' in globals()) or
        (PROCESSED_DIR / 'weather_features_era5.parquet').exists() or
        (PROCESSED_DIR / 'weather_features.parquet').exists()
    )
    
    if weather_already_fetched:
        print("="*80)
        print("WEATHER DATA ALREADY FETCHED")
        print("="*80)
        print("✓ Weather features were already fetched in STEP 2 cell.")
        print("  Loading from memory or saved file...")
        
        # Try to load from memory first
        if 'weather_df_all' in locals() or 'weather_df_all' in globals():
            weather_df_all = locals().get('weather_df_all') or globals().get('weather_df_all')
            print(f"✓ Loaded weather features from memory ({len(weather_df_all)} records)")
        else:
            # Load from saved file
            weather_path = PROCESSED_DIR / 'weather_features_era5.parquet'
            if not weather_path.exists():
                weather_path = PROCESSED_DIR / 'weather_features.parquet'
            if weather_path.exists():
                weather_df_all = pd.read_parquet(weather_path)
                print(f"✓ Loaded weather features from {weather_path} ({len(weather_df_all)} records)")
            else:
                print("⚠ Weather file not found")
        print("="*80)
    
    # Check which weather source to use (only if not already fetched)
    elif USE_GEE_WEATHER and 'ee' in globals():
        # Use GEE ERA5 for weather data (no rate limits, better quality)
        print("="*80)
        print("FETCHING ERA5 WEATHER DATA FROM GOOGLE EARTH ENGINE")
        print("="*80)
        
        print(f"\nConfiguration:")
        print(f"  Product: {GEE_WEATHER_PRODUCT}")
        print(f"  Variables: {len(GEE_WEATHER_BANDS)}")
        print(f"  Lookback: {WEATHER_LOOKBACK_DAYS} days")
        print(f"  Total locations: {len(fire_data)}")
        
        # Load ERA5-Land daily aggregated data
        era5 = ee.ImageCollection(GEE_WEATHER_PRODUCT)
        
        weather_features = []
        
        print(f"\nProcessing weather data...")
        
        for idx, row in fire_data.iterrows():
            if idx % 100 == 0:
                print(f"  Processing {idx+1}/{len(fire_data)}...")
            
            lat = row['LATITUDE']
            lon = row['LONGITUDE']
            fire_date = pd.to_datetime(row['ACQ_DATE'])
            
            # Calculate date range
            start_date = (fire_date - timedelta(days=WEATHER_LOOKBACK_DAYS)).strftime('%Y-%m-%d')
            end_date = fire_date.strftime('%Y-%m-%d')
            
            # Filter ERA5 for this location and time period
            weather = era5.filterDate(start_date, end_date) \
                .select(GEE_WEATHER_BANDS)
            
            # Define point
            point = ee.Geometry.Point([lon, lat])
            
            try:
                # Extract values at point
                weather_series = weather.getRegion(point, 11132).getInfo()  # 11km scale for ERA5-Land
                
                # Parse the results
                if len(weather_series) > 1:
                    headers = weather_series[0]
                    data_rows = weather_series[1:]
                    
                    # Convert to DataFrame
                    weather_df = pd.DataFrame(data_rows, columns=headers)
                    
                    if not weather_df.empty:
                        # Convert time column to datetime
                        weather_df['time'] = pd.to_datetime(weather_df['time'], unit='ms')
                        
                        # Calculate aggregated features
                        feature_dict = {
                            'fire_index': idx,
                            'latitude': lat,
                            'longitude': lon,
                            'fire_date': fire_date
                        }
                        
                        # Add mean, max, min for each band
                        for band in GEE_WEATHER_BANDS:
                            if band in weather_df.columns:
                                feature_dict[f'{band}_mean'] = weather_df[band].mean()
                                feature_dict[f'{band}_max'] = weather_df[band].max()
                                feature_dict[f'{band}_min'] = weather_df[band].min()
                        
                        weather_features.append(feature_dict)
                        
            except Exception as e:
                if idx % 100 == 0:
                    print(f"    Warning: Failed to fetch weather for fire {idx}: {str(e)[:50]}")
                continue
        
        print(f"\n✓ Retrieved weather data for {len(weather_features)}/{len(fire_data)} locations")
        
        # Convert to DataFrame
        if weather_features:
            weather_df_all = pd.DataFrame(weather_features)
            print(f"\n✓ Weather features:")
            print(f"  Features: {len(weather_df_all.columns)}")
            print(f"  Sample count: {len(weather_df_all)}")
            
            # Save weather features
            output_path = PROCESSED_DIR / 'weather_features_era5.parquet'
            weather_df_all.to_parquet(output_path, index=False)
            print(f"✓ Saved to {output_path}")
        else:
            print(f"✗ No weather features extracted")
        
        print("="*80)
        
    elif USE_NASA_POWER:
        # Use NASA POWER API (legacy - has rate limits)
        try:
            from data_ingest.nasa_power.get_humidity import fetch_fire_prediction_weather_batch_sync
            import time
            from datetime import timedelta
            
            print("Fetching weather features from NASA POWER (concurrent)...")
            print(f"  Rate limit: {REQUESTS_PER_HOUR} requests/hour")
            print(f"  Concurrent requests: {MAX_CONCURRENT}")
            
            # Estimate time
            total_requests = len(fire_data)
            effective_rate = REQUESTS_PER_HOUR * MAX_CONCURRENT
            est_hours = total_requests / effective_rate
            print(f"  Total locations: {total_requests}")
            print(f"  Estimated time: {est_hours:.1f} hours ({est_hours*60:.0f} minutes)")
            print()
            
            # Prepare batch data
            locations = []
            start_dates = []
            end_dates = []
            
            for idx, row in fire_data.iterrows():
                fire_date = pd.to_datetime(row['ACQ_DATE'])
                start_date = (fire_date - timedelta(days=DAYS_BEFORE_FIRE)).strftime('%Y%m%d')
                end_date = fire_date.strftime('%Y%m%d')
                
                locations.append({
                    'latitude': row['LATITUDE'],
                    'longitude': row['LONGITUDE'],
                    'fire_index': idx,
                    'fire_date': fire_date
                })
                start_dates.append(start_date)
                end_dates.append(end_date)
            
            # Process in batches to show progress and manage memory
            batch_size = 50  # Process 50 locations at a time
            all_results = []
            start_time = time.time()
            
            for batch_start in range(0, len(locations), batch_size):
                batch_end = min(batch_start + batch_size, len(locations))
                batch_locations = locations[batch_start:batch_end]
                batch_start_dates = start_dates[batch_start:batch_end]
                batch_end_dates = end_dates[batch_start:batch_end]
                
                print(f"  Processing batch {batch_start}-{batch_end-1}...")
                
                # Fetch batch concurrently
                batch_results = fetch_fire_prediction_weather_batch_sync(
                    locations=batch_locations,
                    start_dates=batch_start_dates,
                    end_dates=batch_end_dates,
                    max_concurrent=MAX_CONCURRENT,
                    units='metric',
                    temporal='daily'  # Use daily data for efficiency
                )
                
                all_results.extend(batch_results)
                
                # Show progress
                elapsed = time.time() - start_time
                completed = batch_end
                if completed > 0:
                    rate = completed / elapsed * 3600  # requests per hour
                    remaining = total_requests - completed
                    eta_hours = remaining / rate if rate > 0 else 0
                    success_count = sum(1 for r in all_results if r.get('success', False))
                    print(f"    Status: {completed}/{total_requests} ({rate:.0f} req/hr, {success_count} success, ETA: {eta_hours:.1f}h)")
            
            # Small delay between batches to be respectful
            if batch_end < len(locations):
                time.sleep(API_DELAY)
        
        except ImportError as e:
            print(f"✗ Error importing weather module: {e}")
            print("  Make sure data_ingest.nasa_power.get_humidity is available")
            print("  Try: pip install aiohttp nest-asyncio")
        except Exception as e:
            print(f"✗ Error fetching weather features: {e}")
            import traceback
            traceback.print_exc()
            
            # Process results into features
            print("\nProcessing weather data into features...")
        weather_features = []
        
        
        for result in all_results:
            if result.get('success', False) and result.get('data') is not None:
                df = result['data']
                loc = next((l for l in locations if l['fire_index'] == result.get('fire_index')), None)
                
                if loc and len(df) > 0:
                    weather_feature = {
                        'fire_index': result.get('fire_index'),
                        'latitude': result['latitude'],
                        'longitude': result['longitude'],
                        'fire_date': loc['fire_date'],
                        # Humidity features (use min/max from API)
                        'humidity_mean': df['RH2M'].mean() if 'RH2M' in df.columns else None,
                        'humidity_min': df['RH2M_MIN'].min() if 'RH2M_MIN' in df.columns else df['RH2M'].min() if 'RH2M' in df.columns else None,
                        'humidity_max': df['RH2M_MAX'].max() if 'RH2M_MAX' in df.columns else df['RH2M'].max() if 'RH2M' in df.columns else None,
                        # Precipitation features
                        'precip_total': df['PRECTOT'].sum() if 'PRECTOT' in df.columns else None,
                        'precip_max': df['PRECTOT'].max() if 'PRECTOT' in df.columns else None,
                        'rain_days': (df['PRECTOT'] > 0.1).sum() if 'PRECTOT' in df.columns else None,
                        # Wind features (use max from API)
                        'wind_mean': df['WS10M'].mean() if 'WS10M' in df.columns else None,
                        'wind_max': df['WS10M_MAX'].max() if 'WS10M_MAX' in df.columns else df['WS10M'].max() if 'WS10M' in df.columns else None,
                        # Temperature features (use min/max from API)
                        'temp_mean': df['T2M'].mean() if 'T2M' in df.columns else None,
                        'temp_min': df['T2M_MIN'].min() if 'T2M_MIN' in df.columns else df['T2M'].min() if 'T2M' in df.columns else None,
                        'temp_max': df['T2M_MAX'].max() if 'T2M_MAX' in df.columns else df['T2M'].max() if 'T2M' in df.columns else None,
                    }
                    weather_features.append(weather_feature)
            else:
                # Log failed fetch
                if not result.get('success', False):
                    print(f"  Warning: Failed to fetch fire {result.get('fire_index')}: {result.get('error')}")
        
                    # Summary
            total_time = time.time() - start_time
            print(f"\n✓ Completed in {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")                                                               
            print(f"  Effective rate: {len(all_results)/total_time*3600:.0f} requests/hour")                                                                   
            print(f"  Success rate: {len(weather_features)}/{len(all_results)} ({len(weather_features)/len(all_results)*100:.1f}%)")
            
            # Create DataFrame
            weather_df_all = pd.DataFrame(weather_features)

            if len(weather_df_all) > 0:
                # Save weather features
                output_path = PROCESSED_DIR / 'weather_features.parquet'       
                weather_df_all.to_parquet(output_path, index=False)
                print(f"✓ Saved weather features for {len(weather_df_all)} locations to {output_path}")
            else:
                print("✗ No weather features were successfully fetched")
    else:
        print("⏭ Weather fetching disabled. Set USE_GEE_WEATHER=True or USE_NASA_POWER=True to enable.")
else:
    print("⏭ Skipping weather features (FETCH_WEATHER=False)")


FETCHING ERA5 WEATHER DATA FROM GOOGLE EARTH ENGINE

Configuration:
  Product: ECMWF/ERA5_LAND/DAILY_AGGR
  Variables: 8
  Lookback: 14 days
  Total locations: 1000

Processing weather data...
  Processing 1/1000...
  Processing 101/1000...
  Processing 201/1000...
  Processing 301/1000...
  Processing 401/1000...
  Processing 501/1000...
  Processing 601/1000...
  Processing 701/1000...
  Processing 801/1000...


KeyboardInterrupt: 

## Step 2.5: Validate Humidity and Rain Data Availability

Ensure that a good portion of samples have humidity and rain data, as these are critical features.

In [ ]:
# Validate humidity and rain data availability
if FETCH_WEATHER and 'weather_df_all' in locals() and weather_df_all is not None:
    print("=" * 60)
    print("HUMIDITY AND RAIN DATA VALIDATION")
    print("=" * 60)
    
    # Check for humidity columns (RH2M or similar)
    humidity_cols = [c for c in weather_df_all.columns if 'humidity' in c.lower() or 'rh' in c.lower()]
    rain_cols = [c for c in weather_df_all.columns if 'rain' in c.lower() or 'precip' in c.lower() or 'pr' in c.lower()]
    
    print(f"\nFound humidity columns: {humidity_cols}")
    print(f"Found rain/precipitation columns: {rain_cols}")
    
    if humidity_cols:
        humidity_coverage = (weather_df_all[humidity_cols].notna().any(axis=1)).sum() / len(weather_df_all) * 100
        print(f"\nHumidity data coverage: {humidity_coverage:.1f}%")
    else:
        print("⚠️  No humidity columns found!")
        humidity_coverage = 0
    
    if rain_cols:
        rain_coverage = (weather_df_all[rain_cols].notna().any(axis=1)).sum() / len(weather_df_all) * 100
        print(f"Rain/precipitation data coverage: {rain_coverage:.1f}%")
    else:
        print("⚠️  No rain/precipitation columns found!")
        rain_coverage = 0
    
    # Target: at least 80% coverage for both
    min_coverage = 80
    if humidity_coverage < min_coverage or rain_coverage < min_coverage:
        print(f"\n⚠️  WARNING: Coverage below {min_coverage}% target!")
        print("   Consider fetching more data or checking API responses.")
    else:
        print(f"\n✓ Good coverage: Both humidity and rain data above {min_coverage}%")
    
    # Filter to ensure we keep samples with humidity/rain data
    if humidity_cols and rain_cols:
        has_humidity = weather_df_all[humidity_cols].notna().any(axis=1)
        has_rain = weather_df_all[rain_cols].notna().any(axis=1)
        has_both = has_humidity & has_rain
        
        print(f"\nSamples with both humidity and rain: {has_both.sum()} ({has_both.sum()/len(weather_df_all)*100:.1f}%)")
        
        # Keep samples that have at least humidity OR rain (prefer both)
        valid_weather_indices = weather_df_all[has_humidity | has_rain].index
        print(f"Samples with at least humidity OR rain: {len(valid_weather_indices)} ({len(valid_weather_indices)/len(weather_df_all)*100:.1f}%)")
        
        # Update fire_data to only include samples with weather data
        if len(valid_weather_indices) < len(fire_data):
            print(f"\n⚠️  Filtering fire_data from {len(fire_data)} to {len(valid_weather_indices)} samples with weather data")
            # Map weather indices back to fire_data indices
            weather_fire_indices = weather_df_all.loc[valid_weather_indices, 'fire_index'].values
            fire_data = fire_data[fire_data.index.isin(weather_fire_indices)].reset_index(drop=True)
            print(f"✓ Updated fire_data to {len(fire_data)} samples")
else:
    print("⏭ Skipping humidity/rain validation (weather data not available)")

HUMIDITY AND RAIN DATA VALIDATION

Found humidity columns: []
Found rain/precipitation columns: []
⚠️  No humidity columns found!
⚠️  No rain/precipitation columns found!

⚠️  WARNING: Coverage below 80% target!
   Consider fetching more data or checking API responses.


#Step 3: Fetching Geospatial Data

In [ ]:
if FETCH_TERRAIN:
    # Check if terrain data was already fetched
    terrain_already_fetched = (
        ('terrain_df' in locals() or 'terrain_df' in globals()) or
        (PROCESSED_DIR / 'terrain_features.parquet').exists()
    )
    
    if terrain_already_fetched:
        print("="*80)
        print("TERRAIN DATA ALREADY FETCHED")
        print("="*80)
        print("✓ Terrain features were already fetched.")
        print("  Loading from memory or saved file...")
        
        # Try to load from memory first
        if 'terrain_df' in locals() or 'terrain_df' in globals():
            terrain_df = locals().get('terrain_df') or globals().get('terrain_df')
            print(f"✓ Loaded terrain features from memory ({len(terrain_df)} records)")
        else:
            # Load from saved file
            terrain_path = PROCESSED_DIR / 'terrain_features.parquet'
            if terrain_path.exists():
                terrain_df = pd.read_parquet(terrain_path)
                print(f"✓ Loaded terrain features from {terrain_path} ({len(terrain_df)} records)")
            else:
                print("⚠ Terrain file not found")
        print("="*80)
    
    # Use GEE for terrain (recommended - no rate limits)
    elif USE_GEE_TERRAIN and 'ee' in globals():
        try:
            from data_ingest.google_gee.get_terrain_features import get_terrain_features
            
            print("="*80)
            print("FETCHING TERRAIN FEATURES FROM GOOGLE EARTH ENGINE")
            print("="*80)
            print("Using SRTM DEM via GEE (no rate limits, faster processing)")
            print(f"Processing {len(fire_data)} locations...")
            print("="*80)
            
            terrain_features = []
            
            for idx, row in fire_data.iterrows():
                if idx % 100 == 0:
                    print(f"  Processing {idx+1}/{len(fire_data)}...")
                
                try:
                    # Fetch terrain features from GEE
                    terrain = get_terrain_features(
                        latitude=row['LATITUDE'],
                        longitude=row['LONGITUDE'],
                        scale_meters=30.0,  # 30m resolution
                        buffer_km=1.0,  # 1km buffer
                        fire_date=row['ACQ_DATE'].strftime('%Y-%m-%d') if 'ACQ_DATE' in row else None
                    )
                    
                    terrain['fire_index'] = idx
                    terrain_features.append(terrain)
                    
                except Exception as e:
                    if idx % 100 == 0:
                        print(f"  Warning: Error processing fire {idx}: {e}")
                    # Add NaN values for this location
                    terrain_features.append({
                        'fire_index': idx,
                        'latitude': row['LATITUDE'],
                        'longitude': row['LONGITUDE'],
                        'elevation': np.nan,
                        'elevation_std': np.nan,
                        'slope': np.nan,
                        'slope_max': np.nan,
                        'ruggedness': np.nan,
                        'curvature': np.nan,
                        'canyons': np.nan
                    })
                    continue
            
            # Create DataFrame
            terrain_df = pd.DataFrame(terrain_features)
            
            if len(terrain_df) > 0:
                # Save terrain features
                output_path = PROCESSED_DIR / 'terrain_features.parquet'
                terrain_df.to_parquet(output_path, index=False)
                print(f"✓ Saved terrain features for {len(terrain_df)} locations to {output_path}")
            else:
                print("✗ No terrain features were successfully fetched")
            
        except ImportError as e:
            print(f"✗ Error importing GEE terrain module: {e}")
            print("  Make sure data_ingest.google_gee.get_terrain_features is available")
            print("  Falling back to NASA DEM API...")
            USE_GEE_TERRAIN = False  # Fall back
        except Exception as e:
            print(f"✗ Error fetching terrain features from GEE: {e}")
            import traceback
            traceback.print_exc()
            print("  Falling back to NASA DEM API...")
            USE_GEE_TERRAIN = False  # Fall back
    
    # Use NASA DEM API (legacy - has rate limits)
    if not terrain_already_fetched and not (USE_GEE_TERRAIN and 'ee' in globals()):
        try:
            from data_ingest.nasa_dem.get_terrain import fetch_terrain_features
            
            print("="*80)
            print("FETCHING TERRAIN FEATURES FROM NASA DEM (OpenTopography API)")
            print("="*80)
            print("⚠ Note: NASA DEM API has rate limits. Consider using GEE instead (USE_GEE_TERRAIN=True)")
            print(f"Processing {len(fire_data)} locations...")
            print("="*80)
            
            terrain_features = []
            
            for idx, row in fire_data.iterrows():
                if idx % 10 == 0:
                    print(f"  Processing {idx+1}/{len(fire_data)}...")
                
                try:
                    # Fetch terrain features
                    terrain = fetch_terrain_features(
                        latitude=row['LATITUDE'],
                        longitude=row['LONGITUDE'],
                        buffer_degrees=0.01,  # ~1km buffer
                        compute_derivatives=True
                    )
                    
                    terrain['fire_index'] = idx
                    terrain_features.append(terrain)
                    
                    # Rate limiting
                    time.sleep(API_DELAY)
                
            except Exception as e:
                print(f"  Warning: Error processing fire {idx}: {e}")
                # Add NaN values for this location
                terrain_features.append({
                    'fire_index': idx,
                    'latitude': row['LATITUDE'],
                    'longitude': row['LONGITUDE'],
                    'elevation': np.nan,
                    'elevation_std': np.nan,
                    'slope': np.nan,
                    'slope_max': np.nan,
                    'ruggedness': np.nan,
                    'curvature': np.nan,
                    'canyons': np.nan
                })
                continue
        
        # Create DataFrame
        terrain_df = pd.DataFrame(terrain_features)
        
        if len(terrain_df) > 0:
            # Save terrain features
            output_path = PROCESSED_DIR / 'terrain_features.parquet'
            terrain_df.to_parquet(output_path, index=False)
            print(f"✓ Saved terrain features for {len(terrain_df)} locations to {output_path}")
        else:
            print("✗ No terrain features were successfully fetched")
            
    except ImportError as e:
        print(f"✗ Error importing terrain module: {e}")
        print("  Make sure data_ingest.nasa_dem.get_terrain is available")
        print("  Note: Requires NASA_DEM_API_KEY environment variable")
    except Exception as e:
        print(f"✗ Error fetching terrain features: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭ Skipping terrain features (FETCH_TERRAIN=False)")


Fetching terrain features from NASA DEM...
  Processing 1/1000...


## Step 4: Fetch Geospatial Features (Google Earth Engine)

Fetch geospatial features for each fire detection location:
- Distance to nearest water body
- Forest types (one-hot categorical)
- Fuel layers (one-hot categorical)


In [ ]:
if FETCH_WATER_DISTANCE or FETCH_VEGETATION:
    try:
        # ========================================================================
        # Google Earth Engine Credentials Setup
        # ========================================================================
        import os
        import json
        from pathlib import Path
        
        # Load .env file
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        
        # Get GEE_KEY (filepath to JSON)
        gee_key_path = os.getenv('GEE_KEY')
        
        if not gee_key_path:
            raise ValueError("GEE_KEY environment variable not set")
        
        # Clean up the path - remove quotes and control characters
        # Strip quotes if present
        gee_key_path = gee_key_path.strip('"\'')
        # Remove control characters (form feeds, newlines, etc.)
        gee_key_path = ''.join(c for c in gee_key_path if ord(c) >= 32 or c in '\\/')
        # Normalize path separators
        gee_key_path = gee_key_path.replace('/', '\\')
        gee_key_path = gee_key_path.strip()
        
        # If path doesn't exist, try to find the JSON file automatically
        if not os.path.isfile(gee_key_path):
            print(f"⚠ File does not exist at cleaned path, searching project directory...")
            # Try to find the JSON file in the project directory
            project_root = Path.cwd()
            json_files = list(project_root.glob('*.json'))
            if json_files:
                for jf in json_files:
                    # Check if it looks like a service account key (has client_email field)
                    try:
                        with open(jf, 'r') as f:
                            test_data = json.load(f)
                            if 'client_email' in test_data and 'private_key' in test_data:
                                print(f"✓ Found valid service account key: {jf}")
                                gee_key_path = str(jf.resolve())
                                break
                    except:
                        continue
        
        if not os.path.isfile(gee_key_path):
            raise FileNotFoundError(f"Cannot find GEE credentials file: {gee_key_path}")
        
        print(f"✓ Using GEE credentials: {gee_key_path}")
        
        # Read JSON from file
        try:
            with open(gee_key_path, 'r') as f:
                key_data = json.load(f)
            
            # Check required fields
            required = ['type', 'project_id', 'private_key_id', 'private_key', 'client_email']
            missing = [f for f in required if f not in key_data]
            if missing:
                raise ValueError(f"Missing required fields in credentials: {missing}")
                
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON in credentials file: {e}")
        except Exception as e:
            raise ValueError(f"Error reading credentials file: {e}")
        
        # Initialize Earth Engine
        import ee
        
        # Create credentials using email from JSON
        service_account_email = key_data['client_email']
        credentials = ee.ServiceAccountCredentials(service_account_email, gee_key_path)
        
        # Initialize
        ee.Initialize(credentials)
        print(f"✓ Earth Engine initialized with service account: {service_account_email}")
        
        # ========================================================================
        # Import GEE modules
        # ========================================================================
        from data_ingest.google_gee.get_water_distance import get_distance_to_water
        from data_ingest.google_gee.get_vegetation_features import (
            get_forest_types, 
            get_fuel_layers
        )
        
        print("Fetching geospatial features from Google Earth Engine...")
        geospatial_features = []
        
        for idx, row in fire_data.iterrows():
            if idx % 10 == 0:
                print(f"  Processing {idx+1}/{len(fire_data)}...")
            
            try:
                fire_date = pd.to_datetime(row['ACQ_DATE'])
                fire_date_str = fire_date.strftime('%Y-%m-%d')
                
                geospatial_feature = {
                    'fire_index': idx,
                    'latitude': row['LATITUDE'],
                    'longitude': row['LONGITUDE'],
                    'fire_date': fire_date_str
                }
                
                # Fetch water distance
                if FETCH_WATER_DISTANCE:
                    try:
                        distance = get_distance_to_water(
                            latitude=row['LATITUDE'],
                            longitude=row['LONGITUDE'],
                            max_search_radius_km=50.0,  # Search within 50km
                            scale_meters=100.0,  # 100m resolution for faster processing
                            fire_date=fire_date_str
                        )
                        geospatial_feature['distance_to_water_meters'] = distance
                    except Exception as e:
                        print(f"    Warning: Error fetching water distance for fire {idx}: {e}")
                        geospatial_feature['distance_to_water_meters'] = np.nan
                
                # Fetch vegetation features
                if FETCH_VEGETATION:
                    try:
                        # Get forest types
                        forest_types = get_forest_types(
                            latitude=row['LATITUDE'],
                            longitude=row['LONGITUDE'],
                            fire_date=fire_date_str,
                            data_source='MODIS',
                            classification_scheme='IGBP',
                            scale_meters=250.0
                        )
                        # Add forest type features (excluding metadata)
                        for key, value in forest_types.items():
                            if key not in ['latitude', 'longitude', 'fire_date', 'data_source']:
                                geospatial_feature[f'forest_{key}'] = value
                        
                        # Get fuel layers
                        fuel_layers = get_fuel_layers(
                            latitude=row['LATITUDE'],
                            longitude=row['LONGITUDE'],
                            fire_date=fire_date_str,
                            scale_meters=250.0
                        )
                        # Add fuel layer features (excluding metadata)
                        for key, value in fuel_layers.items():
                            if key not in ['latitude', 'longitude', 'fire_date', 'data_source']:
                                geospatial_feature[f'fuel_{key}'] = value
                                
                    except Exception as e:
                        print(f"    Warning: Error fetching vegetation for fire {idx}: {e}")
                
                geospatial_features.append(geospatial_feature)
                
                # Rate limiting
                time.sleep(API_DELAY)
                
            except Exception as e:
                print(f"  Warning: Error processing fire {idx}: {e}")
                continue
        
        # Create DataFrame
        geospatial_df = pd.DataFrame(geospatial_features)
        
        if len(geospatial_df) > 0:
            # Save geospatial features
            output_path = PROCESSED_DIR / 'geospatial_features.parquet'
            geospatial_df.to_parquet(output_path, index=False)
            print(f"✓ Saved geospatial features for {len(geospatial_df)} locations to {output_path}")
        else:
            print("✗ No geospatial features were successfully fetched")
            
    except ImportError as e:
        print(f"✗ Error importing GEE modules: {e}")
        print("  Make sure data_ingest.google_gee modules are available")
        print("  Note: Requires SERVICE_ACCOUNT and GEE_KEY environment variables")
    except Exception as e:
        print(f"✗ Error fetching geospatial features: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭ Skipping geospatial features (FETCH_WATER_DISTANCE=False and FETCH_VEGETATION=False)")


✓ Using GEE credentials: C:\Users\Drewo\OneDrive\Documents\GIT\fire_prediction\fireprediction-483622-3e2ab1191a16.json
✓ Earth Engine initialized with service account: acount-1@fireprediction-483622.iam.gserviceaccount.com
Fetching geospatial features from Google Earth Engine...
  Processing 1/100...


c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MCD12Q1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MCD12Q1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MCD12Q1

  warnings.warn(warning, category=DeprecationWarning)
c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MOD13Q1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MOD13Q1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD13Q1

  warnings.warn(warning, category=DeprecationWarning)
c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required

  Processing 11/100...
  Processing 21/100...
  Processing 31/100...
  Processing 41/100...
  Processing 51/100...
  Processing 61/100...
  Processing 71/100...
  Processing 81/100...
  Processing 91/100...
✓ Saved geospatial features for 100 locations to data\processed\geospatial_features.parquet


## Step 4.5: Validate Biome Diversity

Ensure diverse biome representation in the dataset to avoid homogeneous biome classification.

In [ ]:
# Validate biome diversity
if FETCH_VEGETATION and 'geospatial_df' in locals() and geospatial_df is not None:
    print("=" * 60)
    print("BIOME DIVERSITY VALIDATION")
    print("=" * 60)
    
    # Find vegetation/biome columns
    veg_cols = [c for c in geospatial_df.columns if any(term in c.lower() for term in ['veg', 'forest', 'biome', 'vegetation', 'grassland', 'shrub', 'savanna'])]
    
    print(f"\nFound vegetation/biome columns: {veg_cols[:10]}..." if len(veg_cols) > 10 else f"\nFound vegetation/biome columns: {veg_cols}")
    
    if veg_cols:
        # Count unique biome combinations
        biome_combinations = geospatial_df[veg_cols].apply(lambda x: '_'.join(x.astype(str)), axis=1)
        unique_biomes = biome_combinations.nunique()
        
        print(f"\nUnique biome combinations: {unique_biomes}")
        print(f"Total samples: {len(geospatial_df)}")
        
        # Check distribution
        biome_counts = biome_combinations.value_counts()
        print(f"\nTop 10 biome types:")
        for biome, count in biome_counts.head(10).items():
            print(f"  {biome[:50]}: {count} samples ({count/len(geospatial_df)*100:.1f}%)")
        
        # Check for homogeneity (if one biome dominates >50%)
        max_biome_pct = biome_counts.iloc[0] / len(geospatial_df) * 100
        
        if max_biome_pct > 50:
            print(f"\n⚠️  WARNING: Biome homogeneity detected!")
            print(f"   Most common biome represents {max_biome_pct:.1f}% of samples")
            print(f"   Target: No single biome should exceed 30-40%")
            print(f"\n   Recommendation: Consider stratified sampling by biome type")
        else:
            print(f"\n✓ Good biome diversity: No single biome exceeds 50%")
            print(f"   Most common biome: {max_biome_pct:.1f}%")
        
        # If we need more diversity, suggest resampling
        if unique_biomes < 5:
            print(f"\n⚠️  WARNING: Low biome diversity ({unique_biomes} unique biomes)")
            print(f"   Target: At least 5-10 different biome types")
            print(f"   Consider expanding geographic sampling to include more diverse regions")
        else:
            print(f"\n✓ Adequate biome diversity: {unique_biomes} unique biome types")
    else:
        print("⚠️  No vegetation/biome columns found in geospatial data!")
        print("   This may indicate an issue with vegetation feature fetching.")
else:
    print("⏭ Skipping biome validation (vegetation data not available)")

## Step 5: Combine All Features

Combine all feature sets into a single dataset ready for machine learning.


In [ ]:
print("Combining all features...")

# Start with fire detection data (base dataset)
combined_df = fire_data.copy()
combined_df['fire_index'] = combined_df.index

# Merge weather features - try from memory first, then from saved parquet
if FETCH_WEATHER:
    weather_loaded = False
    if 'weather_df_all' in locals() and weather_df_all is not None:
        try:
            weather_merge = weather_df_all.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
            combined_df = combined_df.merge(weather_merge, on='fire_index', how='left')
            print(f"✓ Merged weather features from memory ({len(weather_df_all)} records)")
            weather_loaded = True
        except Exception as e:
            print(f"⚠ Error merging weather from memory: {e}")
    
    # Try loading from saved parquet file
    if not weather_loaded:
        weather_parquet = PROCESSED_DIR / 'weather_features.parquet'
        if weather_parquet.exists():
            try:
                weather_df_loaded = pd.read_parquet(weather_parquet)
                weather_merge = weather_df_loaded.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
                combined_df = combined_df.merge(weather_merge, on='fire_index', how='left')
                print(f"✓ Merged weather features from parquet ({len(weather_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading weather parquet: {e}")
        else:
            print(f"⚠ Weather parquet not found: {weather_parquet}")

# Merge terrain features - try from memory first, then from saved parquet
if FETCH_TERRAIN:
    terrain_loaded = False
    if 'terrain_df' in locals() and terrain_df is not None:
        try:
            terrain_merge = terrain_df.drop(columns=['latitude', 'longitude'], errors='ignore')
            combined_df = combined_df.merge(terrain_merge, on='fire_index', how='left')
            print(f"✓ Merged terrain features from memory ({len(terrain_df)} records)")
            terrain_loaded = True
        except Exception as e:
            print(f"⚠ Error merging terrain from memory: {e}")
    
    # Try loading from saved parquet file
    if not terrain_loaded:
        terrain_parquet = PROCESSED_DIR / 'terrain_features.parquet'
        if terrain_parquet.exists():
            try:
                terrain_df_loaded = pd.read_parquet(terrain_parquet)
                terrain_merge = terrain_df_loaded.drop(columns=['latitude', 'longitude'], errors='ignore')
                combined_df = combined_df.merge(terrain_merge, on='fire_index', how='left')
                print(f"✓ Merged terrain features from parquet ({len(terrain_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading terrain parquet: {e}")
        else:
            print(f"⚠ Terrain parquet not found: {terrain_parquet}")

# Merge geospatial features - try from memory first, then from saved parquet
if FETCH_WATER_DISTANCE or FETCH_VEGETATION:
    geospatial_loaded = False
    if 'geospatial_df' in locals() and geospatial_df is not None:
        try:
            geospatial_merge = geospatial_df.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
            combined_df = combined_df.merge(geospatial_merge, on='fire_index', how='left')
            print(f"✓ Merged geospatial features from memory ({len(geospatial_df)} records)")
            geospatial_loaded = True
        except Exception as e:
            print(f"⚠ Error merging geospatial from memory: {e}")
    
    # Try loading from saved parquet file
    if not geospatial_loaded:
        geospatial_parquet = PROCESSED_DIR / 'geospatial_features.parquet'
        if geospatial_parquet.exists():
            try:
                geospatial_df_loaded = pd.read_parquet(geospatial_parquet)
                geospatial_merge = geospatial_df_loaded.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
                combined_df = combined_df.merge(geospatial_merge, on='fire_index', how='left')
                print(f"✓ Merged geospatial features from parquet ({len(geospatial_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading geospatial parquet: {e}")
        else:
            print(f"⚠ Geospatial parquet not found: {geospatial_parquet}")

# Remove fire_index column (no longer needed)
combined_df = combined_df.drop(columns=['fire_index'], errors='ignore')

# Save combined dataset
output_path = PROCESSED_DIR / 'combined_features.parquet'
combined_df.to_parquet(output_path, index=False)
print(f"\n✓ Saved combined features dataset ({len(combined_df)} rows, {len(combined_df.columns)} columns)")
print(f"  File size: {output_path.stat().st_size / (1024**2):.2f} MB")

# Display summary
print("\n=== Dataset Summary ===")
print(f"Total rows: {len(combined_df)}")
print(f"Total columns: {len(combined_df.columns)}")
print(f"\nColumn categories:")
print(f"  - Fire detection: {len([c for c in combined_df.columns if c in ['LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'CONFIDENCE', 'FRP']])}")
print(f"  - Weather: {len([c for c in combined_df.columns if 'rh2m' in c.lower() or 'precipitation' in c.lower() or 'wind' in c.lower() or 'temperature' in c.lower()])}")
print(f"  - Terrain: {len([c for c in combined_df.columns if c in ['elevation', 'slope', 'ruggedness', 'curvature', 'canyons']])}")
print(f"  - Geospatial: {len([c for c in combined_df.columns if 'distance_to_water' in c.lower() or 'forest' in c.lower() or 'fuel' in c.lower()])}")

print(f"\n✓ Data ingestion complete!")
print(f"  Combined dataset saved to: {output_path}")


Combining all features...
✓ Merged weather features from parquet (100 records)
✓ Merged terrain features from parquet (100 records)
✓ Merged geospatial features from memory (100 records)

✓ Saved combined features dataset (100 rows, 82 columns)
  File size: 0.07 MB

=== Dataset Summary ===
Total rows: 100
Total columns: 82

Column categories:
  - Fire detection: 5
  - Weather: 10
  - Terrain: 5
  - Geospatial: 51

✓ Data ingestion complete!
  Combined dataset saved to: data\processed\combined_features.parquet


In [ ]:
# ============================================================================
# VALIDATION: Display sample data from each feature category
# ============================================================================

print("=" * 80)
print("VALIDATION: Sample Data from Each Feature Category")
print("=" * 80)

# Define column categories
fire_detection_cols = [c for c in combined_df.columns if c in [
    'LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'SCAN', 'TRACK', 
    'ACQ_DATE', 'ACQ_TIME', 'SATELLITE', 'CONFIDENCE', 'VERSION', 
    'BRIGHT_T31', 'FRP', 'DAYNIGHT', 'geometry'
]]

weather_cols = [c for c in combined_df.columns if any(x in c.lower() for x in [
    'rh2m', 'precipitation', 'wind', 'temperature', 't2m', 'prectotcorr',
    'ws2m', 'ps', 'humidity', 'rain', 'temp'
])]

terrain_cols = [c for c in combined_df.columns if c.lower() in [
    'elevation', 'elevation_std', 'slope', 'slope_max', 
    'ruggedness', 'curvature', 'canyons'
]]

geospatial_cols = [c for c in combined_df.columns if any(x in c.lower() for x in [
    'distance_to_water', 'forest', 'fuel', 'vegetation', 'ndvi', 'evi', 'npp'
])]

# 1. Fire Detection Features
print("\n" + "=" * 40)
print("1. FIRE DETECTION FEATURES")
print("=" * 40)
print(f"Columns ({len(fire_detection_cols)}): {fire_detection_cols}")
if fire_detection_cols:
    display(combined_df[fire_detection_cols].head())
else:
    print("⚠ No fire detection columns found")

# 2. Weather Features
print("\n" + "=" * 40)
print("2. WEATHER FEATURES")
print("=" * 40)
print(f"Columns ({len(weather_cols)}): {weather_cols}")
if weather_cols:
    display(combined_df[weather_cols].head())
else:
    print("⚠ No weather columns found")

# 3. Terrain Features
print("\n" + "=" * 40)
print("3. TERRAIN FEATURES")
print("=" * 40)
print(f"Columns ({len(terrain_cols)}): {terrain_cols}")
if terrain_cols:
    display(combined_df[terrain_cols].head())
else:
    print("⚠ No terrain columns found")

# 4. Geospatial Features
print("\n" + "=" * 40)
print("4. GEOSPATIAL FEATURES")
print("=" * 40)
print(f"Columns ({len(geospatial_cols)}): {geospatial_cols}")
if geospatial_cols:
    display(combined_df[geospatial_cols].head())
else:
    print("⚠ No geospatial columns found")

# Summary statistics
print("\n" + "=" * 40)
print("SUMMARY: Missing Values by Category")
print("=" * 40)
for name, cols in [
    ("Fire Detection", fire_detection_cols),
    ("Weather", weather_cols),
    ("Terrain", terrain_cols),
    ("Geospatial", geospatial_cols)
]:
    if cols:
        missing = combined_df[cols].isnull().sum().sum()
        total = len(combined_df) * len(cols)
        pct = (missing / total * 100) if total > 0 else 0
        print(f"  {name}: {missing}/{total} missing values ({pct:.1f}%)")
    else:
        print(f"  {name}: No columns")


VALIDATION: Sample Data from Each Feature Category

1. FIRE DETECTION FEATURES
Columns (14): ['LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'SCAN', 'TRACK', 'ACQ_DATE', 'ACQ_TIME', 'SATELLITE', 'CONFIDENCE', 'VERSION', 'BRIGHT_T31', 'FRP', 'DAYNIGHT', 'geometry']


,LATITUDE,LONGITUDE,BRIGHTNESS,SCAN,TRACK,ACQ_DATE,ACQ_TIME,SATELLITE,CONFIDENCE,VERSION,BRIGHT_T31,FRP,DAYNIGHT,geometry
0,8.03751,17.22174,324.88,1.40,1.17,2026-01-05,1340,A,34,6.1NRT,310.28,15.70,D,POINT (17.22174 8.03751)
1,5.93590,21.27689,319.83,1.00,1.00,2026-01-05,1337,A,72,6.1NRT,305.11,9.50,D,POINT (21.27689 5.9359)
2,5.29111,30.14096,313.53,2.62,1.55,2026-01-05,0735,T,56,6.1NRT,298.90,19.45,D,POINT (30.14096 5.29111)
3,8.92654,22.14497,341.46,1.06,1.03,2026-01-05,1340,A,89,6.1NRT,309.59,33.09,D,POINT (22.14497 8.92654)
4,15.34759,20.22583,326.58,1.02,1.01,2026-01-05,1340,A,62,6.1NRT,306.42,9.95,D,POINT (20.22583 15.34759)



2. WEATHER FEATURES
Columns (10): ['rh2m_mean', 'precipitation_total', 'wind_speed_mean', 'temperature_mean', 'rh2m_3day_mean', 'precipitation_3day_total', 'rh2m_14day_mean', 'precipitation_14day_total', 'wind_extreme_12h', 'precipitation_extreme_12h']


,rh2m_mean,precipitation_total,wind_speed_mean,temperature_mean,rh2m_3day_mean,precipitation_3day_total,rh2m_14day_mean,precipitation_14day_total,wind_extreme_12h,precipitation_extreme_12h
0,23.839222,NaN,1.852750,29.058389,23.065417,NaN,23.839222,NaN,2.38,NaN
1,42.518250,NaN,1.235417,28.283667,31.903611,NaN,42.518250,NaN,1.94,NaN
2,43.649500,NaN,1.628361,27.992444,33.835694,NaN,43.649500,NaN,2.53,NaN
3,21.867806,NaN,2.208250,28.242167,21.247639,NaN,21.867806,NaN,3.46,NaN
4,15.066111,NaN,4.185639,24.764417,15.292917,NaN,15.066111,NaN,5.53,NaN



3. TERRAIN FEATURES
Columns (7): ['elevation', 'elevation_std', 'slope', 'slope_max', 'ruggedness', 'curvature', 'canyons']


,elevation,elevation_std,slope,slope_max,ruggedness,curvature,canyons
0,404.499774,5.141390,2.039148,10.766813,4.204556,0.269794,0.0
1,512.131482,10.741735,2.938156,9.500520,5.863655,0.281707,0.0
2,637.042141,6.049330,2.709164,9.595970,6.350739,0.079243,0.0
3,536.686397,10.242034,4.140984,25.416286,5.667468,0.231215,0.0
4,409.512551,2.314409,2.955788,9.940180,4.754941,0.263471,0.0



4. GEOSPATIAL FEATURES
Columns (51): ['distance_to_water_meters', 'forest_forest_type_evergreen_needleleaf_forest', 'forest_forest_type_evergreen_broadleaf_forest', 'forest_forest_type_deciduous_needleleaf_forest', 'forest_forest_type_deciduous_broadleaf_forest', 'forest_forest_type_mixed_forests', 'forest_forest_type_closed_shrublands', 'forest_forest_type_open_shrublands', 'forest_forest_type_woody_savannas', 'forest_forest_type_savannas', 'forest_forest_type_grasslands', 'forest_forest_type_permanent_wetlands', 'forest_forest_type_croplands', 'forest_forest_type_urban_built_up', 'forest_forest_type_cropland_natural_mosaic', 'forest_forest_type_snow_ice', 'forest_forest_type_barren', 'forest_forest_type_water', 'forest_landcover_class_raw', 'fuel_fuel_load', 'fuel_fuel_model', 'fuel_canopy_height', 'fuel_canopy_cover', 'fuel_surface_fuel_load', 'fuel_crown_fuel_load', 'fuel_fuel_model_1', 'fuel_fuel_model_2', 'fuel_fuel_model_3', 'fuel_fuel_model_4', 'fuel_fuel_model_5', 'fuel_fuel_

,distance_to_water_meters,forest_forest_type_evergreen_needleleaf_forest,forest_forest_type_evergreen_broadleaf_forest,forest_forest_type_deciduous_needleleaf_forest,forest_forest_type_deciduous_broadleaf_forest,forest_forest_type_mixed_forests,forest_forest_type_closed_shrublands,forest_forest_type_open_shrublands,forest_forest_type_woody_savannas,forest_forest_type_savannas,...,fuel_fuel_load_high,fuel_canopy_height_low,fuel_canopy_height_medium,fuel_canopy_height_high,fuel_canopy_cover_sparse,fuel_canopy_cover_moderate,fuel_canopy_cover_dense,fuel_has_surface_fuel,fuel_has_crown_fuel,forest_forest_type_unclassified
0,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
1,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
2,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
3,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
4,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN



SUMMARY: Missing Values by Category
  Fire Detection: 0/1400 missing values (0.0%)
  Weather: 400/1000 missing values (40.0%)
  Terrain: 0/700 missing values (0.0%)
  Geospatial: 99/5100 missing values (1.9%)


## Summary and Next Steps

### Data Stores Created

1. **Raw Data**: `data/raw/fire_detections.parquet`
   - Original MODIS fire detection data

2. **Processed Features**:
   - `data/processed/weather_features.parquet` - Weather features from NASA POWER
   - `data/processed/terrain_features.parquet` - Terrain features from NASA DEM
   - `data/processed/geospatial_features.parquet` - Water distance and vegetation from GEE

3. **Combined Dataset**: `data/processed/combined_features.parquet`
   - All features merged together, ready for ML

### Follow-up Questions

1. **Sample Size**: Did you process enough fire detections? Consider increasing `SAMPLE_SIZE` for more robust training data.

2. **Feature Engineering**: The current implementation includes basic aggregations. Consider adding:
   - 3-day consecutive dry/wet/humid indices (normalized)
   - 14-day fuel conditioning index (weighted sum)
   - Weighted weather extremes with exponential weighting
   - Soft binary threshold (humidity/wetness vs temperature)

3. **Temporal Features**: Add circular encoding for:
   - Season (sin/cos of day of year)
   - Time of day (sin/cos of hour)

4. **Data Quality**: Check for:
   - Missing values
   - Outliers
   - Feature distributions

5. **API Credentials**: Ensure you have:
   - NASA POWER API key (optional, but recommended)
   - NASA DEM API key (required for terrain features)
   - Google Earth Engine credentials (SERVICE_ACCOUNT and GEE_KEY)

### Next Steps

1. **Feature Engineering**: Implement the advanced feature calculations mentioned in the README
2. **Data Validation**: Check data quality and handle missing values
3. **Exploratory Data Analysis**: Visualize feature distributions and correlations
4. **Model Training**: Use the combined dataset for ML model training


This file exists to be used for the data ingestion in this project